In [1]:
# Cell 1 - Road model setup
from pathlib import Path
from ultralytics import YOLO

HDD    = Path("/media/hudaifa/Ventoy/datasets")
RDD    = HDD / "rdd4_yolo"                       # converted road dataset (YOLO format)
RUNS   = Path.home() / "drone_ws/yolo/runs"
MODELS = Path.home() / "drone_ws/yolo/models"
ROAD_CLASSES = ["longitudinal_crack", "transverse_crack", "alligator_crack", "pothole"]
print("dataset folder:", RDD)

dataset folder: /media/hudaifa/Ventoy/datasets/rdd4_yolo


In [3]:
%%writefile /home/hudaifa/drone_ws/scripts/rdd2yolo.py
#!/usr/bin/env python3
"""
rdd2yolo.py - convert RDD2022 (Pascal VOC XML, 7 countries) into one YOLO road-damage dataset.

  python rdd2yolo.py                         # defaults below
  python rdd2yolo.py --max-side 1280 --workers 8

Keeps the 4 standard road-damage classes:
  D00 longitudinal_crack   D10 transverse_crack   D20 alligator_crack   D40 pothole
(other RDD labels, e.g. repairs / manholes / blurred markings, are not damage -> ignored, counted in the summary)
Only the labelled 'train' folders are used; each country is split 80 / 10 / 10 (train / val / test), seed 0.
Images are shrunk to --max-side (long side) to save disk and loading time - YOLO trains at 640 anyway.
Safe to run again: finished images are skipped.
Output: <out>/images/{train,val,test}, labels/..., data.yaml, data_test_<country>.yaml
"""
import argparse
import random
import xml.etree.ElementTree as ET
from collections import Counter
from multiprocessing import Pool
from pathlib import Path

import cv2
import yaml
from tqdm import tqdm

HDD = Path("/media/hudaifa/Ventoy/datasets")
CLASSES = {"D00": 0, "D10": 1, "D20": 2, "D40": 3}
NAMES = ["longitudinal_crack", "transverse_crack", "alligator_crack", "pothole"]
SPLIT = (0.8, 0.1, 0.1)


def find_pairs(src):
    pairs = {}
    for train in sorted(src.glob("*_x/*/train")):
        country = train.parent.name
        imgs = {p.stem: p for p in (train / "images").glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")}
        xmls = sorted((train / "annotations").rglob("*.xml"))
        pairs[country] = [(imgs[x.stem], x) for x in xmls if x.stem in imgs]
    return pairs


def convert(job):
    img_p, xml_p, out, split, max_side = job
    stem = img_p.stem
    dst_img = out / "images" / split / f"{stem}.jpg"
    dst_lab = out / "labels" / split / f"{stem}.txt"
    root = ET.parse(xml_p).getroot()
    objs = [(o.findtext("name", "").strip(), o.find("bndbox")) for o in root.iter("object")]
    kept, other = Counter(), Counter()
    if dst_img.exists() and dst_lab.exists():                      # already done (re-run)
        for name, _ in objs:
            (kept if name in CLASSES else other)[name] += 1
        return split, kept, other, 0
    img = cv2.imread(str(img_p))
    if img is None:
        return split, kept, other, 1
    h, w = img.shape[:2]                                           # trust the image, not the XML size
    lines = []
    for name, bb in objs:
        if name not in CLASSES or bb is None:
            other[name] += 1
            continue
        x0, y0, x1, y1 = (float(bb.findtext(k, "0")) for k in ("xmin", "ymin", "xmax", "ymax"))
        x0, x1 = sorted((max(0.0, min(w, x0)), max(0.0, min(w, x1))))
        y0, y1 = sorted((max(0.0, min(h, y0)), max(0.0, min(h, y1))))
        if x1 - x0 < 2 or y1 - y0 < 2:
            continue
        kept[name] += 1
        lines.append(f"{CLASSES[name]} {(x0 + x1) / 2 / w:.6f} {(y0 + y1) / 2 / h:.6f} "
                     f"{(x1 - x0) / w:.6f} {(y1 - y0) / h:.6f}")
    s = min(1.0, max_side / max(h, w))
    if s < 1.0:
        img = cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
    cv2.imwrite(str(dst_img), img, [cv2.IMWRITE_JPEG_QUALITY, 90])
    dst_lab.write_text("\n".join(lines))
    return split, kept, other, 0


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--src", type=Path, default=HDD / "rdd2022_raw/RDD2022")
    ap.add_argument("--out", type=Path, default=HDD / "rdd4_yolo")
    ap.add_argument("--max-side", type=int, default=1280)
    ap.add_argument("--workers", type=int, default=8)
    a = ap.parse_args()
    for d in ("images", "labels"):
        for sp in ("train", "val", "test"):
            (a.out / d / sp).mkdir(parents=True, exist_ok=True)

    pairs = find_pairs(a.src)
    if not pairs:
        raise SystemExit(f"!! no */train folders found under {a.src}")
    jobs, per_country = [], {}
    for country, items in pairs.items():
        random.Random(0).shuffle(items)
        n = len(items)
        n_tr, n_va = round(n * SPLIT[0]), round(n * SPLIT[1])
        parts = {"train": items[:n_tr], "val": items[n_tr:n_tr + n_va], "test": items[n_tr + n_va:]}
        per_country[country] = {k: [p[0].stem for p in v] for k, v in parts.items()}
        jobs += [(img, xml, a.out, sp, a.max_side) for sp, v in parts.items() for img, xml in v]
        print(f"{country:16s} {n:6d} labelled images -> train {len(parts['train'])}, "
              f"val {len(parts['val'])}, test {len(parts['test'])}")

    kept, other, bad, per_split = Counter(), Counter(), 0, Counter()
    with Pool(a.workers) as pool:
        for sp, k, o, b in tqdm(pool.imap_unordered(convert, jobs, chunksize=16), total=len(jobs), desc="converting"):
            kept.update(k); other.update(o); bad += b; per_split[sp] += 1

    base = {"path": str(a.out), "train": "images/train", "val": "images/val", "names": NAMES}
    (a.out / "data.yaml").write_text(yaml.safe_dump({**base, "test": "images/test"}, sort_keys=False))
    for country, parts in per_country.items():                     # per-country test sets (e.g. China_Drone)
        files = [str(a.out / "images/test" / f"{s}.jpg") for s in parts["test"]]
        (a.out / f"test_{country}.txt").write_text("\n".join(files))
        (a.out / f"data_test_{country}.yaml").write_text(
            yaml.safe_dump({**base, "test": f"test_{country}.txt"}, sort_keys=False))

    print("\nimages:", dict(per_split), "  unreadable:", bad)
    print("kept boxes:")
    for name, i in CLASSES.items():
        print(f"   {name} {NAMES[i]:20s} {kept[name]:7d}")
    if other:
        print("ignored labels (not one of the 4 damage classes):", dict(other.most_common()))
    print("\ndataset ready:", a.out / "data.yaml")


if __name__ == "__main__":
    main()

Writing /home/hudaifa/drone_ws/scripts/rdd2yolo.py


In [4]:
# Cell 2 - Convert RDD2022 (7 countries, Pascal VOC XML) -> one YOLO dataset on the HDD
!~/drone_ws/venv/bin/python ~/drone_ws/scripts/rdd2yolo.py

China_Drone        2401 labelled images -> train 1921, val 240, test 240
China_MotorBike    1977 labelled images -> train 1582, val 198, test 197
Czech              2829 labelled images -> train 2263, val 283, test 283
India              7706 labelled images -> train 6165, val 771, test 770
Japan             10506 labelled images -> train 8405, val 1051, test 1050
Norway             8161 labelled images -> train 6529, val 816, test 816
United_States      4805 labelled images -> train 3844, val 480, test 481
converting: 100%|████████████████████████| 38385/38385 [51:34<00:00, 12.40it/s]

images: {'train': 30709, 'val': 3839, 'test': 3837}   unreadable: 0
kept boxes:
   D00 longitudinal_crack     26016
   D10 transverse_crack       11830
   D20 alligator_crack        10616
   D40 pothole                 6544
ignored labels (not one of the 4 damage classes): {'D44': 5057, 'D50': 3581, 'Repair': 1046, 'D43': 793, 'D01': 179, 'D11': 45, 'Block crack': 3, 'D0w0': 1}

dataset ready: /media/hu

In [5]:
# Cell 3 - Train the road model: YOLO11s from COCO weights on 30,709 road photos
import shutil
YOLO("yolo11s.pt").train(data=str(RDD / "data.yaml"), imgsz=640, epochs=50, patience=15, batch=16,
                         cache=False, device=0, project=str(RUNS), name="rdd4_y11s", exist_ok=True)
W_ROAD = RUNS / "rdd4_y11s/weights/best.pt"
shutil.copy(W_ROAD, MODELS / "rdd4_y11s.pt")
print("road model saved:", MODELS / "rdd4_y11s.pt")

New https://pypi.org/project/ultralytics/8.4.215 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.174 🚀 Python-3.10.12 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7796MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/media/hudaifa/Ventoy/datasets/rdd4_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mod

In [6]:
# Cell 4 - Road model on unseen test photos: all countries + each country (China_Drone = drone view)
import math
import pandas as pd
m = YOLO(str(MODELS / "rdd4_y11s.pt"))
rows = []
for yml in ["data.yaml"] + sorted(p.name for p in RDD.glob("data_test_*.yaml")):
    r = m.val(data=str(RDD / yml), split="test", imgsz=640, device=0, verbose=False, plots=False)
    ap = dict(zip(r.box.ap_class_index.tolist(), r.box.ap50.tolist()))
    rows.append({"test set": "ALL" if yml == "data.yaml" else yml[len("data_test_"):-5],
                 "mAP50": round(r.box.map50, 3), "mAP50-95": round(r.box.map, 3),
                 **{c: round(ap.get(i, math.nan), 3) for i, c in enumerate(ROAD_CLASSES)}})
road = pd.DataFrame(rows).set_index("test set")
road.to_csv(RUNS / "rdd4_y11s/test_per_country.csv")
road

Ultralytics 8.4.174 🚀 Python-3.10.12 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7796MiB)
YOLO11s summary (fused): 100 layers, 9,414,348 parameters, 0 gradients, 21.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.8±0.7 ms, read: 5.5±3.6 MB/s, size: 119.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /media/hudaifa/Ventoy/datasets/rdd4_yolo/labels/test... 3837 images, 1442 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3837/3837 91.4it/s 42.0s0.1sss
val: New cache created: /media/hudaifa/Ventoy/datasets/rdd4_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 240/240 10.5it/s 22.8s.1ss
                   all       3837       5712      0.637      0.532      0.583      0.301
Speed: 0.2ms preprocess, 5.1ms inference, 0.0ms loss, 0.2ms postprocess per image
Ultralytics 8.4.174 🚀 Python-3.10

,mAP50,mAP50-95,longitudinal_crack,transverse_crack,alligator_crack,pothole
test set,,,,,,
ALL,0.583,0.301,0.585,0.591,0.672,0.482
China_Drone,0.674,0.375,0.786,0.821,0.241,0.848
China_MotorBike,0.929,0.614,0.892,0.905,0.961,0.960
Czech,0.408,0.191,0.594,0.336,0.530,0.175
India,0.405,0.186,0.369,0.130,0.674,0.447
Japan,0.572,0.268,0.526,0.485,0.703,0.572
Norway,0.277,0.123,0.369,0.327,0.311,0.100
United_States,0.562,0.344,0.777,0.667,0.597,0.208
